# ANN Candidate Generation

Build a vector-based candidate retrieval step for the recommendation system.

This notebook:

- loads `data/processed_data.csv`
- creates multi-hot interest vectors
- uses nearest-neighbor search to retrieve similar users
- returns top 100 candidates for a target user
- applies the existing pairwise retrieval score after ANN retrieval
- validates the candidate output


## Why ANN here?

Candidate generation should be fast. Instead of scoring every user pair manually during inference, we can retrieve similar users from a vector index first.

For this notebook, `NearestNeighbors` is used as a simple ANN-style baseline. In production, this can be replaced with FAISS, HNSW, Annoy, or a managed vector database.

## Note on ANN and LazyPredict

ANN-style retrieval is useful here because candidate generation is a retrieval problem. We need to reduce the full user base into a smaller set of likely candidates before ranking them. Even though this dataset has only 100,000 users and a small interest feature space, using nearest-neighbor retrieval gives us a scalable structure for a larger recommendation system.

In this notebook, we use cosine-based nearest-neighbor retrieval over interest vectors. Since this implementation uses `sklearn.NearestNeighbors` with brute-force search, it is not a true approximate ANN engine like FAISS, HNSWLib, or Annoy. It is still useful as a simple prototype because it follows the same retrieval idea: find users with similar interest vectors first, then apply ranking logic afterward.

LazyPredict is not a good fit for A/B testing here because A/B testing requires real user behavior data. For example, we would need clicks, follows, messages, likes, or other engagement outcomes to compare two models fairly.

This dataset does not contain those labels. If we create a synthetic label from our own heuristic score, LazyPredict would only tell us which model best learns our heuristic. That would not prove the recommendations are actually better. It would only validate that the model can copy the scoring rule we already designed.

Because of that, LazyPredict can be used only as a rough supervised-learning experiment, not as real A/B testing or final model selection for this recommendation system.


In [1]:
from pathlib import Path
import json

import pandas as pd
from sklearn.neighbors import NearestNeighbors

PROCESSED_DATA_CANDIDATES = [
    Path("../data/processed_data.csv"),  # when notebook is run from EDA/
    Path("data/processed_data.csv"),     # when notebook is run from repo root
]

PROCESSED_DATA_PATH = next((path for path in PROCESSED_DATA_CANDIDATES if path.exists()), None)
if PROCESSED_DATA_PATH is None:
    raise FileNotFoundError("Could not find processed_data.csv in data/ or ../data/")

PROCESSED_DATA_PATH


PosixPath('../data/processed_data.csv')

## 1. Load processed data

Use the cleaned dataset from preprocessing.

In [2]:
users_df = pd.read_csv(PROCESSED_DATA_PATH, dtype={"user_id": "string"})
users_df["interests"] = users_df["interests_deduped"].apply(json.loads)

print(f"Rows: {len(users_df):,}")
print(f"Columns: {list(users_df.columns)}")
users_df.head()


Rows: 100,000
Columns: ['user_id', 'name', 'gender', 'dob', 'age', 'city', 'country', 'city_country_key', 'interests_raw', 'interests_list', 'interests_deduped', 'interest_count', 'split', 'interests']


,user_id,name,gender,dob,age,city,country,city_country_key,interests_raw,interests_list,interests_deduped,interest_count,split,interests
0,00001,Jesse Lawhorn,Female,1958-10-15,67,Sibolga,Indonesia,"Sibolga, Indonesia","'Movies', 'Fashion', 'Fashion', 'Books'","[""Movies"", ""Fashion"", ""Fashion"", ""Books""]","[""Movies"", ""Fashion"", ""Books""]",3,train,"[Movies, Fashion, Books]"
1,00002,Stacy Payne,Female,2004-07-21,22,Al Abyār,Libya,"Al Abyār, Libya","'Gaming', 'Finance and investments', 'Outdoor ...","[""Gaming"", ""Finance and investments"", ""Outdoor...","[""Gaming"", ""Finance and investments"", ""Outdoor...",4,train,"[Gaming, Finance and investments, Outdoor acti..."
2,00003,Katrina Nicewander,Female,2000-02-07,26,Wādī as Sīr,Jordan,"Wādī as Sīr, Jordan","'DIY and crafts', 'Music', 'Science', 'Fashion'","[""DIY and crafts"", ""Music"", ""Science"", ""Fashion""]","[""DIY and crafts"", ""Music"", ""Science"", ""Fashion""]",4,validation,"[DIY and crafts, Music, Science, Fashion]"
3,00004,Eric Yarbrough,Male,1985-04-14,41,Matera,Italy,"Matera, Italy","'Outdoor activities', 'Cars and automobiles'","[""Outdoor activities"", ""Cars and automobiles""]","[""Outdoor activities"", ""Cars and automobiles""]",2,train,"[Outdoor activities, Cars and automobiles]"
4,00005,Daniel Adkins,Female,1955-09-18,70,Biruaca,Venezuela,"Biruaca, Venezuela","'Politics', 'History'","[""Politics"", ""History""]","[""Politics"", ""History""]",2,train,"[Politics, History]"


## 2. Create interest vectors

Represent each user as a multi-hot vector over the interest vocabulary.

In [3]:
interest_vocab = sorted({interest for interests in users_df["interests"] for interest in interests})
interest_to_index = {interest: idx for idx, interest in enumerate(interest_vocab)}

def build_interest_vector(interests: list[str]) -> list[int]:
    """Convert a user's deduplicated interests into a multi-hot vector."""
    vector = [0] * len(interest_vocab)
    for interest in interests:
        vector[interest_to_index[interest]] = 1
    return vector


interest_vectors = pd.DataFrame(
    users_df["interests"].apply(build_interest_vector).tolist(),
    columns=[f"interest__{interest}" for interest in interest_vocab],
    index=users_df.index,
)

print(f"Interest vocabulary size: {len(interest_vocab)}")
interest_vectors.head()


Interest vocabulary size: 29


,interest__Art,interest__Beauty,interest__Books,interest__Business and entrepreneurship,interest__Cars and automobiles,interest__Cooking,interest__DIY and crafts,interest__Education and learning,interest__Fashion,interest__Finance and investments,...,interest__Outdoor activities,interest__Parenting and family,interest__Pets,interest__Photography,interest__Politics,interest__Science,interest__Social causes and activism,interest__Sports,interest__Technology,interest__Travel
0,0,0,1,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,1,...,1,0,0,0,0,0,0,0,0,1
2,0,0,0,0,0,0,1,0,1,0,...,0,0,0,0,0,1,0,0,0,0
3,0,0,0,0,1,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0


## 3. Fit nearest-neighbor index

Fit a vector search index over user interest vectors.

In [4]:
ann_index = NearestNeighbors(
    n_neighbors=101,  # ask for 101 because the target user itself may appear first
    metric="cosine",
    algorithm="brute",
)

ann_index.fit(interest_vectors)
ann_index


,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",101
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'brute'
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance<https://docs.scipy.org/doc/scipy/reference/spatial.distance.html>`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'cosine'
,"radius radius: float, default=1.0Range of parameter space to use by default for :meth:`radius_neighbors`queries.",1.0
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"p p: float (positive), default=2Parameter for the Minkowski metric fromsklearn.metrics.pairwise.pairwise_distances. When p = 1, this isequivalent to using manhattan_distance (l1), and euclidean_distance(l2) for p = 2. For arbitrary p, minkowski_distance (l_p) is used.",2
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
Name,Type,Value
effective_metric_ effective_metric_: strMetric used to compute distances to neighbors.,str,'cosine'
effective_metric_params_ effective_metric_params_: dictParameters for the metric used to compute distances to neighbors.,dict,{}


## 4. Select target user

Use one test user as a sample inference case.

In [5]:
target_user = users_df[users_df["split"] == "test"].iloc[0]
target_user_id = target_user["user_id"]
target_index = target_user.name

target_user[["user_id", "name", "age", "city", "country", "city_country_key", "interests_deduped", "split"]]


user_id                                            00029
name                                           Rena Wild
age                                                   43
city                                              Motril
country                                            Spain
city_country_key                           Motril, Spain
interests_deduped    ["History", "Books", "Photography"]
split                                               test
Name: 28, dtype: object

## 5. Retrieve ANN candidates

Retrieve nearest users by interest-vector cosine distance, then exclude the target user.

In [6]:
def retrieve_ann_candidates(target_user_id: str, top_k: int = 100) -> pd.DataFrame:
    """
    Retrieve top-k candidate users using nearest-neighbor search over interest vectors.

    This is the fast retrieval stage. The output is a candidate pool, not final
    recommendations. A ranker should still score and order these candidates.
    """
    matches = users_df[users_df["user_id"] == target_user_id]
    if matches.empty:
        raise ValueError(f"Unknown target_user_id: {target_user_id}")

    target_row_index = matches.index[0]
    target_vector = interest_vectors.loc[[target_row_index]]

    distances, indices = ann_index.kneighbors(target_vector, n_neighbors=top_k + 1)
    candidate_indices = [idx for idx in indices[0] if idx != target_row_index][:top_k]
    candidate_distances = [
        distance
        for idx, distance in zip(indices[0], distances[0])
        if idx != target_row_index
    ][:top_k]

    candidates = users_df.loc[candidate_indices].copy().reset_index(drop=True)
    candidates["ann_cosine_distance"] = candidate_distances
    candidates["ann_similarity"] = 1 - candidates["ann_cosine_distance"]
    candidates["target_user_id"] = target_user_id
    candidates["candidate_user_id"] = candidates["user_id"]

    return candidates


ann_candidates = retrieve_ann_candidates(target_user_id, top_k=100)

ann_candidates[
    [
        "target_user_id",
        "candidate_user_id",
        "name",
        "city",
        "country",
        "interests_deduped",
        "ann_cosine_distance",
        "ann_similarity",
    ]
].head(10)


,target_user_id,candidate_user_id,name,city,country,interests_deduped,ann_cosine_distance,ann_similarity
0,00029,74128,Marcia Sweene,Chiba,Japan,"[""Photography"", ""History"", ""Books""]",0.000000,1.000000
1,00029,49817,Ethel Muniz,Bobingen,Germany,"[""History"", ""Books"", ""Photography""]",0.000000,1.000000
2,00029,28264,Rosa Smith,La Unión,El Salvador,"[""Photography"", ""History"", ""Books""]",0.000000,1.000000
3,00029,74160,Marisol Turner,Danville,United States,"[""Photography"", ""Books"", ""History""]",0.000000,1.000000
4,00029,14313,Mark Boudreau,Owensboro,United States,"[""Photography"", ""History"", ""Books""]",0.000000,1.000000
5,00029,14705,Kimberlee Coffman,Thomastown,Australia,"[""History"", ""Photography"", ""Sports"", ""Books""]",0.133975,0.866025
6,00029,02911,Peggy Hopkins,Lakshmīpur,Bangladesh,"[""Books"", ""History"", ""DIY and crafts"", ""Photog...",0.133975,0.866025
7,00029,22806,John Davis,Norristown,United States,"[""Photography"", ""Gardening"", ""Books"", ""History""]",0.133975,0.866025
8,00029,92398,Abel Conda,Pachino,Italy,"[""Photography"", ""Fitness"", ""Books"", ""History""]",0.133975,0.866025
9,00029,24183,Abraham Freeland,Hāthras,India,"[""Technology"", ""Books"", ""History"", ""Photography""]",0.133975,0.866025


## 6. Add pairwise features and baseline score

ANN retrieves candidates by interest vectors. Pairwise features add explainable ranking signals.

In [7]:
def build_pairwise_features(target_user: pd.Series, candidates: pd.DataFrame) -> pd.DataFrame:
    """Add target-candidate comparison features to ANN candidates."""
    target_interests = set(target_user["interests"])
    features = candidates.copy()

    features["candidate_interest_set"] = features["interests"].apply(set)
    features["interest_overlap_count"] = features["candidate_interest_set"].apply(
        lambda candidate_interests: len(target_interests.intersection(candidate_interests))
    )
    features["interest_union_count"] = features["candidate_interest_set"].apply(
        lambda candidate_interests: len(target_interests.union(candidate_interests))
    )
    features["interest_jaccard_similarity"] = (
        features["interest_overlap_count"] / features["interest_union_count"]
    ).fillna(0.0)

    features["same_city"] = (features["city"] == target_user["city"]).astype(int)
    features["same_country"] = (features["country"] == target_user["country"]).astype(int)
    features["same_city_country_key"] = (
        features["city_country_key"] == target_user["city_country_key"]
    ).astype(int)
    features["age_difference"] = (features["age"] - target_user["age"]).abs()

    features["retrieval_score"] = (
        0.65 * features["interest_jaccard_similarity"]
        + 0.15 * features["same_country"]
        + 0.20 * features["same_city_country_key"]
    )

    return features.sort_values(
        by=["retrieval_score", "ann_similarity", "same_city_country_key", "same_country"],
        ascending=[False, False, False, False],
    ).reset_index(drop=True)


ranked_ann_candidates = build_pairwise_features(target_user, ann_candidates)

ranked_ann_candidates[
    [
        "target_user_id",
        "candidate_user_id",
        "name",
        "city",
        "country",
        "interests_deduped",
        "ann_similarity",
        "interest_overlap_count",
        "interest_jaccard_similarity",
        "same_country",
        "same_city_country_key",
        "age_difference",
        "retrieval_score",
    ]
].head(10)


,target_user_id,candidate_user_id,name,city,country,interests_deduped,ann_similarity,interest_overlap_count,interest_jaccard_similarity,same_country,same_city_country_key,age_difference,retrieval_score
0,00029,74128,Marcia Sweene,Chiba,Japan,"[""Photography"", ""History"", ""Books""]",1.000000,3,1.000000,0,0,21,0.650000
1,00029,49817,Ethel Muniz,Bobingen,Germany,"[""History"", ""Books"", ""Photography""]",1.000000,3,1.000000,0,0,23,0.650000
2,00029,28264,Rosa Smith,La Unión,El Salvador,"[""Photography"", ""History"", ""Books""]",1.000000,3,1.000000,0,0,23,0.650000
3,00029,74160,Marisol Turner,Danville,United States,"[""Photography"", ""Books"", ""History""]",1.000000,3,1.000000,0,0,19,0.650000
4,00029,14313,Mark Boudreau,Owensboro,United States,"[""Photography"", ""History"", ""Books""]",1.000000,3,1.000000,0,0,10,0.650000
5,00029,58460,Jared Hines,Vila-real,Spain,"[""Photography"", ""History""]",0.816497,2,0.666667,1,0,16,0.583333
6,00029,14705,Kimberlee Coffman,Thomastown,Australia,"[""History"", ""Photography"", ""Sports"", ""Books""]",0.866025,3,0.750000,0,0,22,0.487500
7,00029,02911,Peggy Hopkins,Lakshmīpur,Bangladesh,"[""Books"", ""History"", ""DIY and crafts"", ""Photog...",0.866025,3,0.750000,0,0,5,0.487500
8,00029,22806,John Davis,Norristown,United States,"[""Photography"", ""Gardening"", ""Books"", ""History""]",0.866025,3,0.750000,0,0,13,0.487500
9,00029,92398,Abel Conda,Pachino,Italy,"[""Photography"", ""Fitness"", ""Books"", ""History""]",0.866025,3,0.750000,0,0,19,0.487500


## 7. Validate ANN candidate output

Confirm that ANN retrieval returns a clean candidate pool.

In [8]:
ann_validation_report = pd.DataFrame(
    [
        {"check": "returns exactly 100 candidates", "passed": len(ranked_ann_candidates) == 100, "details": len(ranked_ann_candidates)},
        {"check": "target user is excluded", "passed": target_user_id not in set(ranked_ann_candidates["candidate_user_id"]), "details": target_user_id},
        {"check": "candidate IDs are unique", "passed": ranked_ann_candidates["candidate_user_id"].is_unique, "details": int(ranked_ann_candidates["candidate_user_id"].duplicated().sum())},
        {"check": "all candidates share at least one interest", "passed": (ranked_ann_candidates["interest_overlap_count"] > 0).all(), "details": int((ranked_ann_candidates["interest_overlap_count"] == 0).sum())},
        {"check": "scores are sorted descending", "passed": ranked_ann_candidates["retrieval_score"].is_monotonic_decreasing, "details": [float(ranked_ann_candidates["retrieval_score"].max()), float(ranked_ann_candidates["retrieval_score"].min())]},
    ]
)

ann_validation_report


,check,passed,details
0,returns exactly 100 candidates,True,100
1,target user is excluded,True,00029
2,candidate IDs are unique,True,0
3,all candidates share at least one interest,True,0
4,scores are sorted descending,True,"[0.65, 0.43333333333333335]"


## 8. Inspect ANN candidate breakdown

Summarize the retrieved candidate pool.

In [9]:
ann_candidate_breakdown = pd.DataFrame(
    [
        {"metric": "min_ann_similarity", "value": float(ranked_ann_candidates["ann_similarity"].min())},
        {"metric": "max_ann_similarity", "value": float(ranked_ann_candidates["ann_similarity"].max())},
        {"metric": "min_retrieval_score", "value": float(ranked_ann_candidates["retrieval_score"].min())},
        {"metric": "max_retrieval_score", "value": float(ranked_ann_candidates["retrieval_score"].max())},
        {"metric": "avg_interest_overlap", "value": float(ranked_ann_candidates["interest_overlap_count"].mean())},
        {"metric": "avg_interest_jaccard", "value": float(ranked_ann_candidates["interest_jaccard_similarity"].mean())},
        {"metric": "same_country_candidates", "value": int(ranked_ann_candidates["same_country"].sum())},
        {"metric": "same_city_country_candidates", "value": int(ranked_ann_candidates["same_city_country_key"].sum())},
    ]
)

ann_candidate_breakdown


,metric,value
0,min_ann_similarity,0.816497
1,max_ann_similarity,1.000000
2,min_retrieval_score,0.433333
3,max_retrieval_score,0.650000
4,avg_interest_overlap,2.110000
5,avg_interest_jaccard,0.688333
6,same_country_candidates,1.000000
7,same_city_country_candidates,0.000000


## Executed ANN baseline summary

- Sample target user: `00029`, from `Motril, Spain`.
- Target interests: `History`, `Books`, `Photography`.
- ANN retrieval returned exactly `100` candidates.
- The target user was excluded from the candidate pool.
- All candidates share at least one interest with the target user.
- ANN similarity ranged from `0.8165` to `1.00`.
- Retrieval score ranged from `0.4333` to `0.65`.
- Average interest overlap was `2.11`.
- `1` candidate was from the same country.
- `0` candidates had the same `city_country_key`.

Current behavior: ANN retrieval is strongly interest-vector-driven. Location is applied after ANN retrieval as a reranking signal, not as part of the vector index.


## ANN retrieval summary

- ANN-style retrieval is used for Stage 1 candidate generation.
- Users are represented with multi-hot interest vectors.
- The ANN candidate pool is still reranked with pairwise interest and location features.
- This is not the final neural ranker. It is the fast retrieval step before ranking.
- `NearestNeighbors` is acceptable for this assessment baseline; FAISS/HNSW would be the production-scale upgrade.
